# Longer context: 5 → 60 minutes of EEG history

In `CNN_walkthrough.ipynb` the biggest gain came from the **context model**, a GRU that reads the
last **5 minutes** of window embeddings. But the preictal period it has to recognise is **30 minutes**
long. This notebook asks: **does a model that sees 15, 30 or 60 minutes of history predict seizures
better?**

Along the way it fixes a subtle problem with how the history was built (section 1), adds a control
model that sees no EEG at all, and repeats every setting with **3 seeds** so differences come with
error bars.

| Section | What happens | Time on the RTX 5090 |
|---|---|---|
| 1 | Why the history must be read from every recorded window | seconds |
| 2 | How the long-context model works (code) | — |
| 3 | Cache the embeddings of every recorded window | ~15 min EEGNet, ~1–2 h ST-GNN (background) |
| 4 | Train one fold and watch it | ~1 min |
| 5 | Full experiment grid (background) | a few hours |
| 6 | Results and conclusions | seconds |

Run the cells top to bottom. New code for this notebook: `src/long_context.py` (model, histories,
training) and `src/stgnn_embeddings.py` (embeds windows with your ST-GNN models from the other repo).

## 0. Setup

In [ ]:
import os, sys, json, time, subprocess, inspect
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Code, Markdown, display

PROCESSED = ROOT / "data/processed_v3"
STGNN_REPO = Path.home() / "seizure_stgnn_v3"
assert PROCESSED.exists(), "run CNN_walkthrough.ipynb section 0 first (it links data/)"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")

def show(obj):
    display(Code(inspect.getsource(obj), language="python"))

## 1. Why the history must be read from every recorded window

Only two kinds of windows have labels: **interictal** (≥ 60 min from any seizure) and **preictal**
(the 30 min before a seizure). Everything else (the seizure itself, the 5 minutes after it, and the
30–60-minute buffers) is never scored.

The 5-minute context model built its history from **labelled windows only**, so the history stops
wherever unlabelled windows were dropped. Those drops sit exactly in the 30 minutes before every
preictal period. So **how much history a window has tells you its label**, without looking at the EEG.

The cell below measures it: the share of windows whose history is shorter than 5, 30 or 60 minutes,
when the history is built (a) from labelled windows only, as before, and (b) from every recorded
window, as a live system would see the EEG.

In [ ]:
from src.data import list_subjects, load_subject
SUBJECTS = list_subjects(PROCESSED)
DATA = {s: load_subject(PROCESSED, s) for s in SUBJECTS}

def history_minutes(d, which):
    meta = np.load(PROCESSED / d.subject / "meta.npz")
    t_all, lab = meta["t_start"], meta["label"]
    t = np.sort(t_all if which == "all" else t_all[(lab == 0) | (lab == 1)])
    brk = np.concatenate([[0], np.where(np.abs(np.diff(t) - 5) > 1e-3)[0] + 1])    # contiguous stretches
    seg_start = t[brk[np.searchsorted(brk, np.searchsorted(t, d.t_start), side="right") - 1]]
    return (d.t_start - seg_start) / 60

rows = []
for which, name in (("labelled", "labelled windows only (old)"), ("all", "every recorded window (new)")):
    h = {c: np.concatenate([history_minutes(d, which)[d.hard == c] for d in DATA.values()]) for c in (0, 1)}
    for c, lab in ((0, "interictal"), (1, "preictal")):
        rows.append(f"| {name} | {lab} | {np.mean(h[c] < 5):.0%} | {np.mean(h[c] < 30):.0%} | {np.mean(h[c] >= 60):.0%} |")
display(Markdown("| History built from | Windows | < 5 min | < 30 min | ≥ 60 min |\n|---|---|---|---|---|\n" + "\n".join(rows)))

With labelled windows only, **every** preictal window has less than 30 minutes of history, while
a third of interictal windows have a full hour: a 30- or 60-minute model would learn "short history
→ seizure coming" instead of anything in the EEG. Even the 5-minute model could partly exploit it
(24% vs. 6% of windows with < 5 min). Reading every recorded window removes most of the difference;
what is left comes from where the CHB-MIT recording files start and stop.

To measure how much a model could gain from history length alone, section 5 trains **control
models that see no EEG at all**, only which parts of the history were recorded. A real model should
clearly beat its control.

## 2. How the long-context model works

1. Every recorded window (not just labelled ones) is embedded by the fold's window encoder (EEGNet,
   or the ST-GNN from the other repo) and cached on disk.
2. For an anchor window at time *t*, the history is cut into **30-second slots** ending at *t*
   (2 slots per minute: 10 for 5 min, 120 for 60 min). Each slot holds the **mean embedding** of the
   windows recorded in it and the **fraction of the slot that was recorded** (0 for a gap).
   Only windows that start at or before *t* are used, so the model never sees the future.
3. A GRU reads the slots oldest to newest and outputs the risk for the anchor window. Training,
   model selection on validation patients and evaluation are exactly as before.

`History.batch` builds a whole batch of histories on the GPU with prefix sums, so 60-minute
histories cost about the same as 5-minute ones:

In [ ]:
from src import long_context as LC
show(LC.History.batch)

## 3. Cache the embeddings of every recorded window

One file per fold and patient: `cache/emb/<encoder>/<fold>/<patient>.npy` (64 numbers per window,
about 2 GB per encoder). Each fold uses its own encoder, the one that never saw that fold's test patient.

* **EEGNet** encoders from this repo (`results/eegnet_lopo`).
* **ST-GNN + bands** encoders from the ST-GNN repo (`~/seizure_stgnn_v3/results/stgnn_bands_lopo`),
  run with that repo's code by `src/stgnn_embeddings.py`.

The cell starts both in the background and skips anything already cached.

In [ ]:
CACHE = {"eegnet_lopo": ROOT / "cache/emb/eegnet_lopo", "stgnn_bands_lopo": ROOT / "cache/emb/stgnn_bands_lopo"}
cache_cmds = [
    "python -m src.long_context cache --processed_dir data/processed_v3 --source results/eegnet_lopo --out cache/emb/eegnet_lopo",
    f"python src/stgnn_embeddings.py --stgnn_repo {STGNN_REPO} --source {STGNN_REPO}/results/stgnn_bands_lopo --out cache/emb/stgnn_bands_lopo",
]
def n_cached(enc):
    return sum(1 for _ in CACHE[enc].glob("*/*.npy")) if CACHE[enc].exists() else 0

os.makedirs("logs", exist_ok=True)
busy = subprocess.run("pgrep -f 'logs/cache_e[m]b.sh'", shell=True, capture_output=True, text=True).stdout.strip()
if all(n_cached(e) >= 24 * 24 for e in CACHE):
    print("all embeddings cached")
elif busy:
    print("caching already running")
else:
    Path("logs/cache_emb.sh").write_text("#!/usr/bin/env bash\n" + f"cd {ROOT}\nexport PATH={Path(sys.executable).parent}:$PATH\n"
                                         + "\n".join(f'{c} || echo "STEP_FAILED: {c[:50]}"' for c in cache_cmds)
                                         + '\necho "CACHE_FINISHED $(date)"\n')
    subprocess.Popen("setsid nohup bash logs/cache_emb.sh > logs/cache_emb.log 2>&1 < /dev/null &", shell=True)
    print("started caching in the background (log: logs/cache_emb.log)")
for e in CACHE:
    print(f"{e:18s}: {n_cached(e)} of {24 * 24} fold × patient files")

## 4. Train one fold and watch it

A 60-minute model on EEGNet embeddings for the fold whose test patient is **chb05**
(needs the EEGNet cache from section 3). Each line is one epoch; `*` marks a new best validation AUC.

In [ ]:
DEMO = "chb05"
assert (CACHE["eegnet_lopo"] / DEMO).exists(), "wait for the EEGNet cache (section 3)"
LC.main(["train", "--processed_dir", "data/processed_v3", "--embeddings", "cache/emb/eegnet_lopo",
         "--folds_from", "results/eegnet_lopo", "--folds", DEMO, "--minutes", "60", "--seed", "1",
         "--out_dir", "results/long_demo/eegnet_60min", "--verbose"])
from src import evaluate as E
E.evaluate("results/long_demo/eegnet_60min")
display(Markdown(open("results/long_demo/eegnet_60min/results.md").read().split("| Subject")[0]))

## 5. Full experiment grid (background)

Leave-one-patient-out, 24 folds per run, every run scored on all seizures and on lead seizures.

| Group | Encoder | History built from | Minutes | Seeds | Runs |
|---|---|---|---|---|---|
| **Control: no EEG** | none (gap pattern only) | labelled / every window | 5, 60 | 1 | 4 |
| Old setting, re-run | EEGNet, ST-GNN + bands | labelled windows | 5 | 3 | 6 |
| **Main** | EEGNet, ST-GNN + bands | every window | 5, 15, 30, 60 | 3 | 24 |

Seeds change the context model's initialisation and sampling; the window encoders are the same
trained models in every run (retraining the ST-GNN 3× would take most of a day). The ST-GNN runs wait
until the ST-GNN cache is complete. If every result already exists the cell does nothing; set
`FORCE_RERUN = True` to train again.

In [ ]:
SEEDS, MINUTES = (1, 2, 3), (5, 15, 30, 60)
ENCODERS = {"eegnet": "eegnet_lopo", "stgnn_bands": "stgnn_bands_lopo"}
FOLDS_FROM = "results/eegnet_lopo"            # identical LOPO folds for every run (same seed / patients)

def run_name(enc, timeline, minutes, seed=None):
    return f"{enc}_{timeline}_{minutes}min" + (f"_s{seed}" if seed else "")

GRID = []   # (name, extra args)
for tl in ("labelled", "all"):
    for m in (5, 60):
        GRID.append((run_name("noeeg", tl, m), f"--embeddings cache/emb/eegnet_lopo --timeline {tl} --minutes {m} --seed 1 --mask_only"))
for enc, cache in ENCODERS.items():
    for s in SEEDS:
        GRID.append((run_name(enc, "labelled", 5, s), f"--embeddings cache/emb/{cache} --timeline labelled --minutes 5 --seed {s}"))
    for m in MINUTES:
        for s in SEEDS:
            GRID.append((run_name(enc, "all", m, s), f"--embeddings cache/emb/{cache} --timeline all --minutes {m} --seed {s}"))

lines = ["#!/usr/bin/env bash", f"cd {ROOT}", f"export PATH={Path(sys.executable).parent}:$PATH",
         'wait_cache() { until [ $(find cache/emb/$1 -name "*.npy" 2>/dev/null | wc -l) -ge 576 ]; do sleep 60; done; }',
         "wait_cache eegnet_lopo"]
for name, extra in GRID:
    out = f"results/long/{name}"
    if "stgnn_bands" in extra:
        lines.append("wait_cache stgnn_bands_lopo")
    lines += [f'if [ ! -f {out}/lead4h/results.json ]; then echo "[$(date +%H:%M)] {name}"',
              f"  python -m src.long_context train --processed_dir data/processed_v3 --folds_from {FOLDS_FROM} --out_dir {out} {extra} || echo STEP_FAILED {name}",
              f"  python -m src.evaluate --results_dir {out} > /dev/null",
              f"  python -m src.evaluate --results_dir {out} --lead_gap_h 4 --processed_dir data/processed_v3 --out_dir {out}/lead4h > /dev/null",
              "fi"]
lines.append('echo "ALL_FINISHED $(date)"')
Path("logs/long_grid.sh").write_text("\n".join(lines) + "\n")

FORCE_RERUN = False
done = [n for n, _ in GRID if Path(f"results/long/{n}/lead4h/results.json").exists()]
running = subprocess.run("pgrep -f 'logs/long_gri[d].sh'", shell=True, capture_output=True, text=True).stdout.strip()
if FORCE_RERUN:
    subprocess.run("rm -rf results/long", shell=True)
    done = []
if len(done) == len(GRID):
    print(f"all {len(GRID)} runs finished")
elif running:
    print(f"grid already running: {len(done)} of {len(GRID)} runs finished")
else:
    subprocess.Popen("setsid nohup bash logs/long_grid.sh > logs/long_grid.log 2>&1 < /dev/null &", shell=True)
    print(f"started {len(GRID) - len(done)} runs in the background (log: logs/long_grid.log)")

In [ ]:
# Progress: re-run any time
done = [n for n, _ in GRID if Path(f"results/long/{n}/lead4h/results.json").exists()]
print(f"{len(done)} of {len(GRID)} runs finished")
log = Path("logs/long_grid.log")
if log.exists():
    print("\n".join(log.read_text().splitlines()[-3:]))

## 6. Results

Each cell of the table is the **mean ± SD over 3 seeds** (controls: 1 seed). *p* is the median over
seeds of the p-value against a random predictor with the same false-alarm rate.
**Lead seizures** start ≥ 4 h after the previous seizure (65 of the 151 test seizures).

In [ ]:
def summ(path):
    p = Path(path) / "results.json"
    return json.loads(p.read_text().replace("NaN", "null"))["summary"] if p.exists() else None

def agg(names, sub=""):
    S = [summ(Path("results/long") / n / sub) for n in names]
    S = [s for s in S if s]
    if not S:
        return None
    f = lambda k: np.array([s[k] for s in S], float)
    return {"n": len(S), "pred": f("predicted"), "sz": S[0]["seizures"], "fa": f("fpr_per_hour"),
            "chance": f("random_predictor_sensitivity"), "p": f("p_value_vs_chance"), "auc": f("window_auc_mean_per_subject")}

def cell(a, auc=False):
    if a is None:
        return "*not run*", "", "", ""
    pm = lambda x, d=1: f"{x.mean():.{d}f}" + (f" ± {x.std():.{d}f}" if len(x) > 1 else "")
    p = np.median(a["p"])
    return (f"{pm(a['pred'])} / {a['sz']}", pm(a["fa"], 2), f"{np.mean(a['chance']):.0%}",
            "< 1e-4" if p < 1e-4 else f"{p:.2g}")

LABEL = {"noeeg": "No EEG (control)", "eegnet": "EEGNet", "stgnn_bands": "ST-GNN + bands"}
def table_rows():
    rows = []
    for tl in ("labelled", "all"):
        for m in (5, 60):
            rows.append(("noeeg", tl, m, [run_name("noeeg", tl, m)]))
    for enc in ENCODERS:
        rows.append((enc, "labelled", 5, [run_name(enc, "labelled", 5, s) for s in SEEDS]))
        for m in MINUTES:
            rows.append((enc, "all", m, [run_name(enc, "all", m, s) for s in SEEDS]))
    return rows

out = ["| Encoder | History from | Minutes | All: predicted | FA/h | chance | p | Lead: predicted | FA/h | chance | p | window AUC |",
       "|---|---|---|---|---|---|---|---|---|---|---|---|"]
for enc, tl, m, names in table_rows():
    a, l = agg(names), agg(names, "lead4h")
    auc = "" if a is None else f"{a['auc'].mean():.3f}"
    out.append(f"| {LABEL[enc]} | {'labelled only' if tl == 'labelled' else 'every window'} | {m} | "
               + " | ".join(cell(a)) + " | " + " | ".join(cell(l)) + f" | {auc} |")
display(Markdown("\n".join(out)))

In [ ]:
# Seizures predicted vs. history length (mean ± SD over seeds), with the no-EEG control and chance
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
colors = {"eegnet": "#3b6ea5", "stgnn_bands": "#2e8b57"}
for ax, sub, title in ((axes[0], "", "all seizures"), (axes[1], "lead4h", "lead seizures only")):
    for enc in ENCODERS:
        ms, mu, sd, ch = [], [], [], []
        for m in MINUTES:
            a = agg([run_name(enc, "all", m, s) for s in SEEDS], sub)
            if a:
                ms.append(m); mu.append(a["pred"].mean() / a["sz"]); sd.append(a["pred"].std() / a["sz"]); ch.append(a["chance"].mean())
        if ms:
            ax.errorbar(ms, mu, yerr=sd, marker="o", capsize=3, color=colors[enc], label=LABEL[enc])
            ax.plot(ms, ch, ls=":", color=colors[enc], alpha=0.7, label=f"chance at {LABEL[enc]}'s alarm rate")
    for m in (5, 60):
        a = agg([run_name("noeeg", "all", m)], sub)
        if a:
            ax.scatter([m], [a["pred"].mean() / a["sz"]], marker="x", s=60, color="0.3",
                       label="no-EEG control" if m == 5 else None, zorder=3)
    ax.set_xscale("log"); ax.set_xticks(MINUTES); ax.set_xticklabels(MINUTES)
    ax.set_xlabel("history (minutes)"); ax.set_ylim(0, 0.8); ax.set_title(f"LOPO, {title}", loc="left")
axes[0].set_ylabel("share of seizures predicted"); axes[1].legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

## 7. Conclusions

*Written once the grid in section 5 has finished.*